In [1]:
import os; os.environ['WORKDIR'] = "/home/choij/workspace/ChargedHiggsAnalysis" 
import sys; sys.path.insert(0, os.environ['WORKDIR'])

import torch
from torch_geometric.data import Data

from ROOT import TFile
from libPython.DataFormat import Particle
from libPython.DataFormat import get_muons, get_electrons, get_jets
from libPython.Selection import select
from libPython.Management import FileManager, MVAManager
from libPython.HistTools import HistogramWriter

Welcome to JupyROOT 6.26/06


In [ ]:
# link files


In [2]:
MASSPOINT = "MHc-130_MA-90"
BACKGROUND = "TTLL_powheg"
f_sig = TFile.Open(f"{os.environ['WORKDIR']}/SelectorOutput/2017/Skim3Mu__/Split/Selector_TTToHcToWAToMuMu_{MASSPOINT}_0.root")
f_bkg = TFile.Open(f"{os.environ['WORKDIR']}/SelectorOutput/2017/Skim3Mu__/Split/Selector_{BACKGROUND}_0.root")

manager = MVAManager()
writer = HistogramWriter(outfile=f"{os.environ['WORKDIR']}/triLepRegion/test.root")

In [3]:
def Loop(evt, manager, writer, writerPrefix):
    muons = get_muons(evt)
    electrons = get_electrons(evt)
    jets, bjets = get_jets(evt)
    METv = Particle(evt.METvPt, 0., evt.METvPhi, 0.)
    
    region = select("3Mu", evt, muons, electrons, jets, bjets, "tight")
    if not region:
        return None
    
    scores = manager.getScores(muons+electrons+jets)
    
    writer.fill_muons(f"{writerPrefix}/{region}/muons", muons, evt.GenWeight*evt.TrigLumi)
    writer.fill_electrons(f"{writerPrefix}/{region}/electrons", electrons, evt.GenWeight*evt.TrigLumi)
    writer.fill_jets(f"{writerPrefix}/{region}/jets", jets, evt.GenWeight*evt.TrigLumi)
    writer.fill_jets(f"{writerPrefix}/{region}/bjets", bjets, evt.GenWeight*evt.TrigLumi)
    writer.fill_object(f"{writerPrefix}/{region}/METv", METv, evt.GenWeight*evt.TrigLumi)
    for classifier, score in scores.items():
        writer.fill_hist(f"{writerPrefix}/{region}/{classifier}/score", score, evt.GenWeight*evt.TrigLumi, 100, 0., 1.)

In [4]:
for evt in f_sig.Events:
    Loop(evt, manager, writer, "signal")
for evt in f_bkg.Events:
    Loop(evt, manager, writer, "background")
f_sig.Close()
f_bkg.Close()
writer.close()

Saving histograms in /home/choij/workspace/ChargedHiggsAnalysis/triLepRegion/test.root...
